In [71]:
import requests
from langchain_core.prompts import PromptTemplate 
from langchain_core.tools import tool, StructuredTool, InjectedToolArg
from langchain_ollama import ChatOllama
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser
from langchain.agents import create_agent
from pydantic import BaseModel, Field
from typing import Optional
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
import os
from dotenv import load_dotenv
import yfinance as yf
import pandas_ta as ta
import json

load_dotenv()

True

In [72]:
NEWS_DATA_API=os.getenv('NEWS_DATA_API')

In [73]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate(
    template="""You are a query parameter builder for the NewsData.io API focusing strictly on Indian stock market catalysts (NSE/BSE).

Target Topic: {topic}

Step 1: Choose your search scope (CHOOSE EXACTLY ONE):
- 'q=...' (Broad search across the whole article)
- 'qInTitle=...' (Strict search only in the headline)
- 'qInMeta=...' (Search titles, URLs, and meta tags)
- 'timeframe=...' (Search by timeframe)
- 'category=...' (Search by category)

CRITICAL: Never combine these. Pick only one.

Step 2: Apply these mandatory rules:
1. Use actionable single-stock catalyst terms (e.g., "quarterly results", "order win", "acquisition").
2. Strictly DO NOT include broad market phrases like "Sensex" or "Nifty".
3. Always append '&category=business&country=in'.

Validation Rules:
- STRICTLY EXCLUDE foreign equitie  s (e.g., Nike, Tesla, Apple, Google). Only include companies actively listed on the NSE or BSE.
- STRICTLY EXCLUDE general editorial comparisons or educational articles without fresh catalysts.
- For each valid event, state the exact NSE/BSE ticker symbol (e.g., UNIONBANK, INFY, WIPRO).

Output Format:
- Output ONLY the final query string separated by '&'.
- Do NOT include the base URL or conversational explanations.
- Example output: qInTitle=order+win+OR+acquisition&category=business&country=in
""",
    input_variables=["topic"]
)

In [74]:
model = ChatOllama(model="gemma4:e4b", temperature=0.7, num_ctx=8192)

In [75]:
parser = StrOutputParser()

In [76]:
class Article(BaseModel):
    title: str = Field(..., description="The title of the news article"),
    description: Optional[str] = Field(..., description="A brief description of the news article"),
    source: Optional[str] = Field(..., description="The source of the news article")

In [77]:
class ArticleList(BaseModel):
    articles: list[Article] = Field(description="List of 5 news articles")

In [78]:
articles_parser = PydanticOutputParser(pydantic_object=ArticleList)


In [79]:
@tool
def generate_query_params(topic: str):
    """Generate a formatted URL query parameter string for fetching news articles."""
    # print("generate_query_params called with topic:", topic)
    model = ChatOllama(model="gemma4:e4b", temperature=0.7, max_tokens=4000)
    chain = prompt | model | parser

    prompt_text = prompt.invoke({'topic': topic})
    print(prompt_text)

    query_params = chain.invoke({'topic' : topic})
    # print(query_params)

    return query_params

In [80]:
class NewsQueryParams(BaseModel):
    topic : str = Field(..., description="The topic to search news articles for")

# @tool
def fetch_news_articles(topic: str):
    """ Tool to fetch news articles. MUST only be called after generating query parameters. """

    query_chain = prompt | model | parser
    query_params = query_chain.invoke({"topic": topic})

    print("query_params_before_fetching: ",query_params)
    
    url = f"https://newsdata.io/api/1/latest?apikey={NEWS_DATA_API}&{query_params}&size=10"
    response = requests.get(url)
    data = response.json()
    
    # 1. Check for API success
    if data.get("status") == "success":
        clean_articles = []
        
        # 2. Extract only the essential text needed by the financial analyst prompt
        for article in data.get("results", []):
            clean_articles.append({
                "title": article.get("title"),
                "description": article.get("description"),
                "source": article.get("source_name")
            })
            
        # 3. Return the minimized list of dictionaries, NOT the raw JSON
        return clean_articles

    return data

fetch_news_articles_tool = StructuredTool.from_function(
    func=fetch_news_articles,
    name="fetch_news_articles",
    description="Fetch relevant news articles by passing relevant query parameters to the API",
    args_schema=NewsQueryParams
)

In [81]:
agent = create_agent(
    model=model,
    tools=[fetch_news_articles_tool],
)

format_instructions = articles_parser.get_format_instructions()

prompt_text = f"""You are an equity research analyst covering individual stocks on the NSE and BSE.

Step 1:
Call `fetch_news_articles` with a query focused on company-specific corporate actions (such as quarterly earnings surprises, mega contract wins, mergers, or regulatory penalties).

Step 2:
From the fetched results, filter out and ignore:
- Broad market index wrap-ups (e.g., Sensex/Nifty up/down commentary).
- Trading holiday calendars and exchange scheduling notices.
- Global macroeconomic recaps (crude oil prices, US bond yields).

Step 3:
Identify 5 specific companies with actionable catalysts capable of moving the individual stock price by 2% or more.

Output strictly according to these formatting instructions:
{format_instructions}"""

response = agent.invoke({"messages": [HumanMessage(content=prompt_text)]})
raw_content = response["messages"][-1].content
# print("Raw content from agent:\n", raw_content)
structured_result = articles_parser.parse(raw_content)

for i, item in enumerate(structured_result.articles):
    print(f"Article {i+1}:\nTitle: {item.title}\nDescription: {item.description}\nSource: {item.source}\n")

c:\YASHIT\AI\market-announcements\.venv\Lib\site-packages\pydantic\json_schema.py:2463: PydanticJsonSchemaWarning: Default value (FieldInfo(annotation=NoneType, required=True, description='The title of the news article'),) is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)
c:\YASHIT\AI\market-announcements\.venv\Lib\site-packages\pydantic\json_schema.py:2463: PydanticJsonSchemaWarning: Default value (FieldInfo(annotation=NoneType, required=True, description='A brief description of the news article'),) is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)


query_params_before_fetching:  qInTitle=(quarterly+results+OR+earnings+surprise+OR+acquisition+OR+merger+OR+"order+win"+OR+"contract+win")&category=business&country=in
Article 1:
Title: EMCOR Group declares $0.40 quarterly dividend
Description: None
Source: Investing India

Article 2:
Title: Tesla quarterly deliveries beat as Europe demand recovers
Description: Tesla beat Wall Street estimates for third-quarter deliveries on Friday as a demand rebound in Europe offset the loss of US tax incentives and stiffer competition in China Shares of the Austin Texas-based company rose more than 3 in early trading Through last close the stock had fallen more than 21 so far this year
Source: Devdiscourse

Article 3:
Title: Brink’s declares quarterly dividend of $0.255 per share
Description: None
Source: Investing India

Article 4:
Title: Danaos declares $1.00 quarterly and $5.00 special dividend
Description: Danaos declares $1.00 quarterly and $5.00 special dividend
Source: Investing India

Articl

In [82]:
# print(clean_json_str)
# article_list = ArticleList.model_validate_json(clean_json_str)
# print(article_list)

In [83]:
import pandas as pd
import requests
import io

def get_valid_nse_tickers():
    """
    Downloads the official active equity master list directly from the NSE 
    and returns a set of valid ticker symbols.
    """
    url = "https://nsearchives.nseindia.com/content/equities/EQUITY_L.csv"
    
    # NSE blocks standard python user agents, so we spoof a standard browser
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
        "Accept": "text/csv,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8"
    }
    
    try:
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        
        # Read the CSV into pandas
        df = pd.read_csv(io.StringIO(response.text))
        
        # The 'SYMBOL' column contains the official tickers (e.g., 'RELIANCE', 'TCS')
        valid_symbols = set(df['SYMBOL'].str.strip().str.upper())
        print(f"Loaded {len(valid_symbols)} active NSE tickers.")
        return valid_symbols
        
    except Exception as e:
        print(f"Failed to fetch NSE master list: {e}")
        # Fallback to a small critical list if the download fails
        return {"RELIANCE", "TCS", "HDFCBANK", "INFY", "ITC"} 

# Run this once globally
VALID_NSE_TICKERS = get_valid_nse_tickers()

Loaded 2593 active NSE tickers.


In [84]:
# %%
class TickerTechnicalScore(BaseModel):
    ticker: str = Field(description="The official NSE ticker symbol, or 'NONE'")
    open: float | None = Field(default=None, description="The opening price")
    close: float | None = Field(default=None, description="The closing price")
    high: float | None = Field(default=None, description="The highest price")
    low: float | None = Field(default=None, description="The lowest price")
    rsi14: float | None = Field(default=None, description="14-day RSI")
    macd12_26_9: float | None = Field(default=None, description="MACD value")
    ema20: float | None = Field(default=None, description="20-day EMA")
    news_title: str = Field(description="Title of the catalyst article")
    conclusion: str = Field(description="Actionable takeaway on stock movement")

class TickersTechnicalReport(BaseModel):
    reports: list[TickerTechnicalScore] = Field(description="List of technical evaluations for the identified stocks")

In [ ]:
@tool
def get_ticker_technicals(ticker: str):
    """Fetches technical indicators (RSI, MACD, EMA) for a given NSE ticker symbol."""
    
    # Clean up what the LLM gave us
    base_ticker = ticker.strip().upper().replace(".NS", "").replace(".BO", "")
    
    # 1. GROUND TRUTH CHECK: Is this a real NSE company?
    if base_ticker not in VALID_NSE_TICKERS:
        return {
            "error": f"'{base_ticker}' is not an active NSE listed equity. Skipping analysis."
        }
    
    # 2. Re-append the .NS suffix for Yahoo Finance
    ticker_clean = f"{base_ticker}.NS"

    try:
        df = yf.Ticker(ticker_clean).history(period="6mo")
        
        # 3. Check if yfinance actually returned data
        if df.empty or len(df) < 30:
            return {"error": f"Insufficient trading data found for {ticker_clean}."}

        # Calculate technical indicators safely
        df.ta.rsi(length=14, append=True)
        df.ta.macd(fast=12, slow=26, signal=9, append=True)
        df.ta.ema(length=20, append=True)

        return {
            "latest_open": round(float(df["Open"].iloc[-1]), 2),
            "latest_close": round(float(df["Close"].iloc[-1]), 2),
            "latest_high": round(float(df["High"].iloc[-1]), 2),
            "latest_low": round(float(df["Low"].iloc[-1]), 2),
            "latest_rsi": round(float(df["RSI_14"].iloc[-1]), 2),
            "latest_macd": round(float(df["MACD_12_26_9"].iloc[-1]), 2),
            "latest_ema20": round(float(df["EMA_20"].iloc[-1]), 2)
        }
    except Exception as e:
        return {"error": f"Failed calculating technicals for {ticker_clean}: {str(e)}"}

In [92]:
agent_technical = create_agent(
    model=model,
    tools=[get_ticker_technicals]
)


ticker_technical_parser = PydanticOutputParser(pydantic_object=TickersTechnicalReport)

articles_text = "\n".join([
    f"- {art.title} (Source: {art.source}): {art.description}"
    for art in structured_result.articles
])

prompt_content = f"""
You are a technical market analyst.
Review the following news articles:

{articles_text}

Task:
Process EVERY single article provided above into your final output. Do not skip any.

For each article, follow this logic:
1. Identify if it explicitly mentions a specific Indian company listed on the NSE or BSE.
2. If YES (Valid Indian Ticker):
   - Extract the official NSE ticker symbol (e.g., RELIANCE, TCS, APLAPOLLO).
   - Call the `get_ticker_technicals` tool for this ticker.
   - Write a conclusion evaluating both the news catalyst and the technical data.
3. If NO (Foreign company, macroeconomic event, or unnamed entity):
   - Set the ticker strictly to the word: NONE
   - DO NOT call the `get_ticker_technicals` tool.
   - Leave the technical fields (open, close, rsi, macd, etc.) as null.
   - Write a conclusion summarizing the news impact on the broader market or sector.

Return your final answer strictly following these formatting instructions:
{ticker_technical_parser.get_format_instructions()}
"""
agent_response = agent_technical.invoke({
    "messages": [HumanMessage(content=prompt_content)]
})

raw_text = agent_response["messages"][-1].content

# Use the Pydantic parser directly to handle clean deserialization
try:
    parsed_report = ticker_technical_parser.parse(raw_text)
    for item in parsed_report.reports:
        print(f"Ticker: {item.ticker}")
        print(f"RSI: {item.rsi14} | Close: ₹{item.close}")
        print(f"Conclusion: {item.conclusion}\n" + "-"*40)
except Exception:
    # Fallback to manual JSON parsing if the local LLM dropped part of the Pydantic format
    clean_json = raw_text.replace("```json", "").replace("```", "").strip()
    data = json.loads(clean_json)
    reports = data.get("reports", data) if isinstance(data, dict) else data
    for item in reports:
        print(f"Ticker: {item.get('ticker')} | Conclusion: {item.get('conclusion')}")

Ticker: NONE
RSI: None | Close: ₹None
Conclusion: The dividend announcement by EMCOR, an international company, provides positive short-term support for its equity but has limited direct impact on the broader Indian market or domestic sectors.
----------------------------------------
Ticker: NONE
RSI: None | Close: ₹None
Conclusion: Tesla's beating delivery estimates, driven by European recovery, signals resilience in the global EV market. This news is generally positive for the broader auto and EV sector, suggesting improving consumer demand despite geopolitical and competitive headwinds.
----------------------------------------
Ticker: NONE
RSI: None | Close: ₹None
Conclusion: The dividend declaration by Brink's, a foreign entity, is a positive signal for income-focused investors but does not serve as a direct catalyst for Indian market movements.
----------------------------------------
Ticker: DANAOS
RSI: None | Close: ₹None
Conclusion: The declaration of a substantial special divi